In [ ]:
from sentence_transformers import SentenceTransformer
import torch
# Load embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("google/embeddinggemma-300m", device=device)

def create_embedding(texts):
    if type(texts) == list: # the input param is a list contains text chunks
        embeddings = model.encode(texts)
        return embeddings
    elif type(texts) == str: # just one text chunk
        embedding = model.encode([texts])
        return embedding
    else:
        raise ValueError("The texts paramters should be str or list[str]")

print(device)

In [ ]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

In [ ]:
#LLM api call
from google import genai
with open(f"{root_path}/API_KEY.txt", "r", encoding="utf-8") as f:
    API_KEY = f.read()

def call_gemini(text):
    client = genai.Client(api_key = API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=text
    )
    return response.text, response.usage_metadata.total_token_count

In [ ]:
import pandas as pd
import numpy as np
import json
medical_questions_answered_loaded = pd.read_parquet("data/medical_questions_answered.parquet")
question_ids = medical_questions_answered_loaded['id'].to_list()
medical_questions_answered_loaded

In [ ]:
import sys
sys.path.append(root_path)
from graphs.Node import Node
from testing.metrics.faithfulness import compute_faithfulness #Detect hallucinations: Measures how much the LLM answer is based on the retrieved context
"""
question: str,
answer: str,
contexts: List[str],
call_gemini,
max_retries: int = 5
"""
from testing.metrics.context_recall import compute_context_recall #Evaluate retrieval coverage: Measures how much of the reference (ground-truth) answer is supported by the retrieved context.
"""
question: str,
contexts: List[str],
reference_answer: str,
call_gemini,
max_retries: int = 5
"""
from testing.metrics.coverage import compute_coverage #Measures how completely a model’s response covers the factual content of a reference (ground-truth) answer.
"""
question: str,
reference: str,
response: str,
call_gemini,
max_retries: int = 5
"""
from testing.metrics.rouge import rouge_scorer
#Calculate F-1 of lexical overlap by LCS: longest common subsequence (in terms of word order, adjacency not required)
#Precision = len(LCS) / len(Answer)
#Recall = len(LCS) / len(Ref)
"""
answer: str,
ground_truth: str,
rouge_type: str = "rougeL",
mode: str = "fmeasure"
"""
from testing.metrics.accuracy import compute_answer_accuracy
"""
question: str,
answer: str,
ground_truth: str,
call_gemini,
create_embedding,
weights: List[float] = [0.75, 0.25],
beta: float = 1.0,
max_retries: int = 5
"""


In [ ]:
try:
    medical_questions_answered = pd.read_parquet("data/medical_questions_answered.parquet")
except:
    medical_questions_answered = medical_questions.copy(deep=True)
    medical_questions_answered[["LLM_answer", "LLM_context", "LLM_tokens"]] = None


medical_questions_answered

In [ ]:
LLM_answer = (
    medical_questions_answered.loc[medical_questions_answered[["LLM_answer", "LLM_context", "LLM_tokens"]].notna().all(axis=1) &
        (medical_questions_answered[["LLM_answer", "LLM_context", "LLM_tokens"]] != "").all(axis=1)
        ].set_index("id")[["LLM_answer", "LLM_context", "LLM_tokens"]].to_dict(orient="index"))

LLM_answer


In [ ]:
from tqdm import tqdm
import time
sep = "\n\n" + "-"*100 + "\n\n"
MAX_RETRIES = 20
SLEEP_SEC = 30

def get_answers():
    for qid in tqdm(question_ids):
        if qid in LLM_answer:
            continue
        row = medical_questions.loc[medical_questions["id"] == qid].iloc[0]
        question = row["question"]
        query_context["question"] = question
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                context = get_context(query_context,graph_context,embedding_context,API_KEY,debug=False)
                full_context = sep.join(context.values())
                prompt = answer_prompt(full_context, question)
                response = call_gemini(prompt)
                answer = response.text
                usage = response.usage_metadata
                #log
                LLM_answer[qid] = {
                    "LLM_answer": answer,
                    "LLM_context": full_context,
                    "LLM_tokens": usage.total_token_count,
                }
                break

            except Exception as e:
                code = e.error.code if hasattr(e, "error") else e
                print(f"[ERROR] qid={qid} failed after {attempt} retries: {code}")
                if attempt == MAX_RETRIES:
                    return
                time.sleep(SLEEP_SEC * attempt)  #backoff

get_answers()


In [ ]:
LLM_answer

In [ ]:
llm_df = (
    pd.DataFrame.from_dict(LLM_answer, orient="index")
      .reset_index()
      .rename(columns={"index": "id"})
)
llm_df

In [ ]:
print("min:", llm_df["LLM_tokens"].min())
print("max:", llm_df["LLM_tokens"].max())
print("avg:", llm_df["LLM_tokens"].mean())
print("sum:", llm_df["LLM_tokens"].sum())